# 📊 Superstore Sales Analysis
**Dataset:** Sample - Superstore.csv  
**Goal:** Explore sales and profit patterns to help managers make data-driven decisions.

---

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Load the dataset
df = pd.read_csv('Sample_-_Superstore.csv', encoding='latin1')

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Dataset information
df.info()

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('=== Missing Values Summary ===')
print(missing_df[missing_df['Missing Count'] > 0] if missing_df['Missing Count'].sum() > 0 else 'No missing values found ✅')

## 2. Data Cleaning

In [ ]:
# --- Step 1: Parse date columns ---
# Order Date and Ship Date are stored as strings; convert to datetime for time-based analysis.
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=False)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=False)

# Extract useful time features
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month
df['Month_Name'] = df['Order Date'].dt.strftime('%b')
df['YearMonth']  = df['Order Date'].dt.to_period('M')

# --- Step 2: Postal Code as string ---
# Postal codes are identifiers, not numeric values; cast to string to avoid accidental arithmetic.
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)

# --- Step 3: Derive Profit Margin ---
# Profit margin = Profit / Sales. Guard against division by zero.
df['Profit Margin'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'], 0)

# --- Step 4: Flag negative-profit rows (not removed — useful for analysis) ---
n_loss = (df['Profit'] < 0).sum()
print(f'Rows with negative profit (loss-making orders): {n_loss:,} ({n_loss/len(df)*100:.1f}%)')

print('\nCleaning complete. Updated dtypes:')
print(df[['Order Date', 'Ship Date', 'Postal Code', 'Profit Margin']].dtypes)

## 3. KPI Calculation

In [ ]:
# --- Core KPIs ---
total_rows    = len(df)
total_revenue = df['Sales'].sum()
total_profit  = df['Profit'].sum()
avg_margin    = df['Profit Margin'].mean()
total_orders  = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()

print('=' * 45)
print('           KEY PERFORMANCE INDICATORS')
print('=' * 45)
print(f'  Total rows (transactions) : {total_rows:>10,}')
print(f'  Unique orders             : {total_orders:>10,}')
print(f'  Unique customers          : {total_customers:>10,}')
print(f'  Total Revenue             : ${total_revenue:>12,.2f}')
print(f'  Total Profit              : ${total_profit:>12,.2f}')
print(f'  Overall Profit Margin     : {avg_margin*100:>10.2f}%')
print('=' * 45)

In [ ]:
# --- Mean & Median for numeric columns ---
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Profit Margin']
stats = df[numeric_cols].agg(['mean', 'median']).round(3)
print('Mean & Median of numeric columns:')
display(stats)

In [ ]:
# --- Top 5 Sub-Categories by total Sales ---
top5_subcat = (df.groupby('Sub-Category')['Sales']
                 .sum()
                 .sort_values(ascending=False)
                 .head(5)
                 .reset_index())
top5_subcat.columns = ['Sub-Category', 'Total Sales']
top5_subcat['Total Sales'] = top5_subcat['Total Sales'].map('${:,.2f}'.format)
print('Top 5 Sub-Categories by Revenue:')
display(top5_subcat)

In [ ]:
# --- Grouped statistics ---

# Average Sales & Profit per Region
region_stats = (df.groupby('Region')[['Sales', 'Profit']]
                  .agg(['mean', 'sum'])
                  .round(2))
region_stats.columns = ['Avg Sales', 'Total Sales', 'Avg Profit', 'Total Profit']
print('Regional Performance:')
display(region_stats)

print()

# Total Revenue per Category
cat_revenue = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print('Total Revenue by Category:')
display(cat_revenue.map('${:,.2f}'.format).rename('Total Sales'))

In [ ]:
# --- Monthly Revenue Trend ---
monthly = (df.groupby(['Year', 'Month', 'Month_Name'])['Sales']
             .sum()
             .reset_index()
             .sort_values(['Year', 'Month']))
print('Monthly Revenue (first 12 rows):')
display(monthly.head(12))

## 4. Data Visualization

In [ ]:
# ── Plot 1: Histogram — Distribution of Sales ──────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

clipped = df['Sales'].clip(upper=df['Sales'].quantile(0.97))   # clip extreme outliers for readability
ax.hist(clipped, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)

mean_val   = df['Sales'].mean()
median_val = df['Sales'].median()
ax.axvline(mean_val,   color='crimson',     linestyle='--', linewidth=1.6, label=f'Mean   ${mean_val:,.0f}')
ax.axvline(median_val, color='darkorange',  linestyle=':',  linewidth=1.6, label=f'Median ${median_val:,.0f}')

ax.set_title('Distribution of Order Sales (clipped at 97th percentile)', fontsize=13, fontweight='bold')
ax.set_xlabel('Sales ($)')
ax.set_ylabel('Frequency')
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('plot1_sales_histogram.png', bbox_inches='tight')
plt.show()
print('The distribution is right-skewed: most orders are small, but a few high-value orders pull the mean up.')

In [ ]:
# ── Plot 2: Bar Chart — Total Sales & Profit by Category ───────────────────
cat_group = df.groupby('Category')[['Sales', 'Profit']].sum().reset_index()

x = np.arange(len(cat_group))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, cat_group['Sales'],  width, label='Sales',  color='#4C72B0')
bars2 = ax.bar(x + width/2, cat_group['Profit'], width, label='Profit', color='#55A868')

ax.set_xticks(x)
ax.set_xticklabels(cat_group['Category'], fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1e6:.1f}M' if y >= 1e6 else f'${y/1e3:.0f}K'))
ax.set_title('Total Sales & Profit by Product Category', fontsize=13, fontweight='bold')
ax.set_ylabel('USD')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'${bar.get_height()/1e3:.0f}K', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'${bar.get_height()/1e3:.0f}K', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('plot2_category_bar.png', bbox_inches='tight')
plt.show()
print('Technology drives the highest profit despite not being the top revenue category by margin ratio.')

In [ ]:
# ── Plot 3: Correlation Heatmap ─────────────────────────────────────────────
corr_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Profit Margin']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 10})
ax.set_title('Correlation Heatmap of Key Numeric Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot3_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Key finding: Discount has a strong negative correlation with Profit Margin.')

In [ ]:
# ── Plot 4: Line Chart — Monthly Revenue Trend ─────────────────────────────
monthly_trend = (df.groupby('YearMonth')['Sales']
                   .sum()
                   .reset_index()
                   .sort_values('YearMonth'))
monthly_trend['YearMonth_str'] = monthly_trend['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly_trend['YearMonth_str'], monthly_trend['Sales'],
        marker='o', markersize=4, linewidth=1.8, color='#4C72B0')
ax.fill_between(monthly_trend['YearMonth_str'], monthly_trend['Sales'], alpha=0.15, color='#4C72B0')

# Mark the peak month
peak_idx = monthly_trend['Sales'].idxmax()
peak_row = monthly_trend.loc[peak_idx]
ax.annotate(f"Peak\n${peak_row['Sales']/1e3:.0f}K",
            xy=(peak_row['YearMonth_str'], peak_row['Sales']),
            xytext=(0, 15), textcoords='offset points',
            ha='center', fontsize=8, color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson', lw=1.2))

step = max(1, len(monthly_trend) // 12)
ax.set_xticks(monthly_trend['YearMonth_str'][::step])
ax.set_xticklabels(monthly_trend['YearMonth_str'][::step], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1e3:.0f}K'))
ax.set_title('Monthly Revenue Trend', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Total Sales ($)')
plt.tight_layout()
plt.savefig('plot4_monthly_trend.png', bbox_inches='tight')
plt.show()
print('Revenue peaks strongly in November–December, suggesting seasonal demand (holiday period).')

In [ ]:
# ── Plot 5: Profit by Region & Segment — Grouped Bar ───────────────
pivot = df.groupby(['Region', 'Segment'])['Profit'].sum().unstack()

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, colormap='tab10', width=0.7, edgecolor='white')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y/1e3:.0f}K'))
ax.set_title('Total Profit by Region and Customer Segment', fontsize=13, fontweight='bold')
ax.set_xlabel('Region')
ax.set_ylabel('Total Profit ($)')
ax.set_xticklabels(pivot.index, rotation=0)
ax.legend(title='Segment')
plt.tight_layout()
plt.savefig('plot6_region_segment_profit.png', bbox_inches='tight')
plt.show()
print('The West region leads in overall profit and dominates in Consumer and Corporate segments; the Central region lags notably behind.')

## 5. Insights

Based on the analysis above, here are the key findings:

1. **Discounts are the #1 profit killer.** The correlation heatmap confirms a strong negative relationship between discount rates and profit margins. This data suggests that the current discount policy is unsustainable and should be revisited urgently to protect the company's bottom line.

2. **The West region is the most profitable.** The West is the most profitable region overall, driven by its dominance in the Consumer and Corporate segments. While the East generates comparable revenue and leads in the Home Office segment, the West's superior performance in the larger segments ensures its position as the top profit contributor.

3. **The Central region underperforms.** Central shows the lowest total profit and may benefit from targeted promotions, pricing adjustments, or a review of the product mix.

4. **Technology has the best profit margin.** While Furniture and Office Supplies generate significant revenue, Technology products produce the highest absolute profit. Investing in Technology inventory and marketing is likely to yield the best ROI.

5. **Furniture is consistently loss-prone.** Several Furniture sub-categories (notably Tables and Bookcases) generate significant losses. The combination of high discounts on Furniture items is a key driver.

6. **Revenue is highly seasonal.** There is a clear spike in Q4 (November–December) every year. The company should plan inventory, logistics, and staffing to meet this seasonal demand.

7. **Sales distribution is heavily right-skewed.** The median order value is significantly lower than the mean, confirming a heavily right-skewed distribution. A small fraction of high-value orders drives a disproportionate share of total revenue, making the retention of top-tier customers a strategic priority.

8. **Consumer segment drives the most orders.** Consumers represent the largest order volume, but Corporate customers tend to have higher average order values — a useful segmentation for targeted sales efforts.

9. **18.7% of all transactions are loss-making.** Nearly 1 in 5 orders generates a negative profit, often correlated with high discount rates. Identifying and managing these orders could immediately improve overall profitability.

10. **Quantity alone does not predict profit.** The near-zero correlation between Quantity and Profit indicates that selling more units per order does not reliably increase profit — product mix and discount rate matter far more.